# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Joy Naa Ayi-Kooley Addy
**Student ID:** 89342028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [1]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [2]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
             temperature=0.7, max_tokens=500):
  response = client.chat.completions.create(
         model=MODEL,
         messages=[
             {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
         temperature=temperature,
         max_tokens=max_tokens,
     )
  print("Token usage is ",response.usage)
  return response.choices[0].message.content
#
# TODO: Call it once with a simple question and print the answer.
answer = ask_llm("Would you rather be stranded in the Amazon or at sea, give one reason")
print(answer)
# TODO: Print response.usage as well — how many tokens did your call consume?

Token usage is  CompletionUsage(completion_tokens=48, prompt_tokens=56, total_tokens=104, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.135218256, prompt_time=0.002944918, completion_time=0.155537855, total_time=0.158482773)
I would rather be stranded in the Amazon. One reason is that the Amazon provides a more stable source of fresh water, which is essential for survival, whereas being at sea can make it difficult to find a reliable source of drinking water.


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**

1. The system role is basically what directs the model on how to behave generally, including persona, constraints and guidelines whilst the user role is what delivers actual tasks which the model answers. An example using a Multilingual translator:
System role: You are an English to Japanese translator, output only the Japanese translation.
User role: Thank you
( The translator outputs the Japanese word for Thank you)

2. A token is an atomic unit which the model reads. It could be a word, character or a subword.
API providers bill per token instead of per request because a request cannot be measured as a fixed unit for computation, where all request s cost the same time to run.

### Part 1.2 — Temperature: the randomness dial

In [3]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
# TODO: Print all 10 answers, grouped by temperature.
question = "Would you rather be stranded in the Amazon or at sea, give one reason"

print(".......TEMPERATURE 0.0........")
for i in range(5):
  answer = ask_llm(question, temperature =0.0, max_tokens=500)
  print(f"{i+1}. {answer}")

print("\n.......TEMPERATURE 1.2 ........")
for i in range(5):
  answer = ask_llm(question, temperature =1.2, max_tokens=500)
  print(f"{i+1}. {answer}")


.......TEMPERATURE 0.0........
Token usage is  CompletionUsage(completion_tokens=44, prompt_tokens=56, total_tokens=100, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.008087172, prompt_time=0.002954241, completion_time=0.132863079, total_time=0.13581732)
1. I would rather be stranded in the Amazon. One reason is that the Amazon has a vast array of freshwater sources, which would provide me with a more reliable means of accessing drinking water, increasing my chances of survival.
Token usage is  CompletionUsage(completion_tokens=44, prompt_tokens=56, total_tokens=100, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.008735407, prompt_time=0.002953691, completion_time=0.134353095, total_time=0.137306786)
2. I would rather be stranded in the Amazon. One reason is that the Amazon has a vast array of freshwater sources, which would provide me with a more reliable means of accessing drinking water, increasing my chances of survival.
Token us

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:**

1. At temperature 0.0, all the responses were the same and chose the Amazon. The same reason was also given, citing access to freshwater.This shows that the model is consistent and predictable.
At temperature 1.2, the model still chose the AM=mazon for all five responses. However, the explanations given were different. Some answers mentioned rivers and streams, whilst others cited food and shelter. The number of completion tokens were also different, This means that the higher temperature produced responses with more variety although the main decision was the same.

For the loan support system, I would use a low temperature. This is because I think that consistency would be one of the most important features of a model for thus type of scenario. The results should be predictable rather than having unpredictable varieties. The model provides decion support whilst the final decision is done by the loan officer.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [5]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize this: "


for letter_id in ["L002","L006"]:
  letter_text = LETTERS[letter_id]

  answer = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{letter_text}")
  print(f"\n...{letter_id}....")
  print(answer)
# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
SUMMARY_SYSTEM_V2= """You are an assistant to a microfinance loan officer.
Summarize loan applications accurately and neutrally.
Use only information stated in the application.
Do not invent, assume, or infer facts that are not provided.
Keep the summary to 3-4 sentences."""

SUMMARY_PROMPT_V2= "Summarize this loan application:\n\n{letter_text}"


for letter_id in ["L002","L006"]:
  letter_text = LETTERS[letter_id]
  answer = ask_llm(
      SUMMARY_PROMPT_V2.format(letter_text=letter_text),
      system_prompt=SUMMARY_SYSTEM_V2,
      temperature=0.0)
  print(f"\n...{letter_id}....")
  print(answer)
# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
output_1 = {}
output_2 = {}

for letter_id in ["L002","L006"]:
  letter_text = LETTERS[letter_id]

  output_1[letter_id] = ask_llm(f"SUMMARY_PROMPT_V2.format{letter_text}")

  output_2[letter_id] = ask_llm(SUMMARY_PROMPT_V2.format(letter_text=letter_text),
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0.0)
for letter_id in ["L002","L006"]:
  print(f"\n{'.'*60}")
  print(f"LETTER {letter_id}")
  print(f"{'.'*60}")

  print("\n... Output 1 (V1): Naive ....")
  print(output_1[letter_id])

  print("\n... Output 2(V2): Structured ")
  print(output_2[letter_id])



Token usage is  CompletionUsage(completion_tokens=75, prompt_tokens=134, total_tokens=209, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.00843267, prompt_time=0.013060553, completion_time=0.266153191, total_time=0.279213744)

...L002....
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow period in business but is optimistic it will improve after the festive season. He has no collateral to offer but is asking for help and promises to repay the loan when he can.
Token usage is  CompletionUsage(completion_tokens=81, prompt_tokens=136, total_tokens=217, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.008424073, prompt_time=0.008375402, completion_time=0.241593637, total_time=0.249969039)

...L006....
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**
1. V1 added its own interpretations which the letters did not state. For example in L006, V1 mentioned "Although he has no experience and no collateral", however, the application did not mention that Kofi has no experience, instead it stated " He has not yet begun any of these ventures". V2 fixes this by saying "He has not yet begun any of these ventures."

2. "no invented details" is an essential instruction because the model supports actual loan decisions. If this instruction does not exist and the model invests information concerning income, repayment ability and so on, the loan officer may make wrong decisions because the information the model supplied was fake.
This type of failure is known as hallucination in LLM literature. It occurs when the model produces plausible but false information. For example: V1 stated that Kofi has no experience , but this was never stated in the letter.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [6]:
import json
import re
import pandas as pd

# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
EXTRACT_PROMPT = """Extract the requested fields from the loan application below.

Return ONLY  a valid JSON object with EXACTLY these keys:
{{
  "applicant_name": "string",
  "amount_ghs": "number",
  "purpose": "string",
  "monthly_profit_ghs":"number or null",
  "has_collateral_or_guarantor": "boolean",
  "repayment_months": "number or null" }}

Rules:
     - If a field is not stated in the letter, use null. Do not guess.
     - temperature=0
     - Use only information explicitly stated in the loan application.
     - Do not guess or infer missing information.
     - amount_ghs and monthly_profit_ghs must be numbers, not strings.
     - repayment_months must be a number or null.
     - has_collateral_or_guarantor must be true if the application explicitly mentions collateral or a guarantor, and false if it explicitly states that there is none.
     - Do not add any keys.
     - Do not include explanations or markdown.


Here is a worked example:

Loan application:
"My name is Jane Doe. I run a small bakery in Tema and I am requesting GHS 6,000 to purchase an oven. My monthly profit is GHS 700. My mother will guarantee the loan. I will repay it over 10 months."

Expected JSON:
{{
  "applicant_name": "Jane Doe",
  "amount_ghs": 6000,
  "purpose": "purchase an oven",
  "monthly_profit_ghs": 700,
  "has_collateral_or_guarantor": true,
  "repayment_months": 10
}}

Now extract the fields from this loan application:

{letter_text}
"""
# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
def extract_fields(letter_text):
  prompt = EXTRACT_PROMPT.format(letter_text = letter_text)

  result = ask_llm(prompt,
                   temperature = 0.0,
                   max_tokens=300)
  try:
    result = re.sub(r"^```json\s*", "", result.strip(), flags=re.IGNORECASE)
    result = re.sub(r"^```\s*", "", result.strip())
    result = re.sub(r"\s*```$", "", result.strip())
    data = json.loads(result)
    return data

  except (json.JSONDecodeError, TypeError) as e:
    print("Error occurred.")
    print("Raw output:", result)
    return None

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
results =[]
for letter_id, letter_text in LETTERS.items():
  extracted = extract_fields(letter_text)
  if extracted is not None:
    extracted["letter_id"] = letter_id
    results.append(extracted)
results_frame = pd.DataFrame(results)
columns = ["letter_id","applicant_name","amount_ghs","purpose","monthly_profit_ghs","has_collateral_or_guarantor","repayment_months"]
results_frame = results_frame[columns]
display(results_frame)

Token usage is  CompletionUsage(completion_tokens=73, prompt_tokens=523, total_tokens=596, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.008305263, prompt_time=0.027694092, completion_time=0.09534257, total_time=0.123036662)
Token usage is  CompletionUsage(completion_tokens=72, prompt_tokens=483, total_tokens=555, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.008392282, prompt_time=0.034009316, completion_time=0.106948788, total_time=0.140958104)
Token usage is  CompletionUsage(completion_tokens=77, prompt_tokens=537, total_tokens=614, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.008271312, prompt_time=0.058657178, completion_time=0.101292642, total_time=0.15994982)
Token usage is  CompletionUsage(completion_tokens=70, prompt_tokens=503, total_tokens=573, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.008248736, prompt_time=0.029336334, completion_time=0.110659306, total_time=0.1

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**

1.  The few-shot example must not come from the six letters being processed because it could lead to a form of bias in the model where it memorises the actual evaluation data.
The use of separate tests would help the model to work optimally on unseen data in the future.


2. "use null, do not guess" instructs the model to return missing values as null(NaN) and without it the model may manufacture false values not stated in the application. For instance, without this instruction, the model may try to estimate profit where the letters do not state one. This prevents misleading evaluations also.


3. A temperature of 0 is good for extraction because of the need for consistent and predictable results. In our loan application, if we want to get the same values in the various fields each time we run then we must avoid higher temperatures(they lead to variation and inconsistency).
However, it is undesirable for creative tasks because they require diverse and varied responses that cannot be achieved at this temperature.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [7]:
import json

# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
BRIEF_PROMPT = """You are an assistant supporting a human microfinance loan officer.
Your task is to review a loan application using BOTH:
1. The original application letter
2. The extracted structured data

Produce a concise decision-support brief with exactly these four sections:
1. Strengths
a. List specific strengths supported by the application.
b.  Do not invent or assume information.
2. Risks / Red Flags
a. List specific risks or concerns supported by the application.
b. Do not make unsupported assumptions.
3. Missing Information
a. List important information or documents the loan officer should request.
b. If something important is not stated, identify it as missing rather than guessing.
4. Suggested Next Step
a. Recommend an appropriate follow-up action such as:
  "invite for interview",
  "request documents",
  "request additional financial information",
  or "flag for senior review".
b. Do NOT recommend "approve" or "reject".

Important:
1. Use only information provided in the letter and extracted data.
2.Do not invent facts.
3.The LLM is providing decision support only.
4.The final lending decision must always be made by a human loan officer.

Original loan application:
{letter_text}

Extracted data:
{extracted_json}

"""
# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

briefs={}

for letter_id, letter_text in LETTERS.items():
  extracted = results_frame[results_frame["letter_id"] == letter_id].iloc[0].to_dict()
  extracted_json = json.dumps(extracted, indent=2, default=str)

  prompt = BRIEF_PROMPT.format(letter_text=letter_text,extracted_json=extracted_json)

  brief = ask_llm(prompt,temperature=0.0,max_tokens=700)

  briefs[letter_id] = brief

for letter_id in ["L001", "L002","L003", "L006"]:
    print("." * 70)
    print(f"LETTER {letter_id}")
    print("." * 70)
    print(briefs[letter_id])
    print()



Token usage is  CompletionUsage(completion_tokens=354, prompt_tokens=501, total_tokens=855, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.036338914, prompt_time=0.027435225, completion_time=1.299865739, total_time=1.327300964)
Token usage is  CompletionUsage(completion_tokens=268, prompt_tokens=456, total_tokens=724, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.270315168, prompt_time=0.023416293, completion_time=0.944725358, total_time=0.968141651)
Token usage is  CompletionUsage(completion_tokens=484, prompt_tokens=519, total_tokens=1003, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.036593261, prompt_time=0.033258618, completion_time=1.47299494, total_time=1.506253558)
Token usage is  CompletionUsage(completion_tokens=333, prompt_tokens=478, total_tokens=811, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.036581197, prompt_time=0.025416548, completion_time=1.114587359, total_ti

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**

1. Yes, the system identified the right strenghts and red flags in each application.
L003: This system identified a good number of strengths. The brief stated that Darko Fshions is a registered business, has 18 months of sales records, earns an average GHS2800 monthly profit and also the fixed deposits that can be used as collateral. It also identifies some risks, such as comparing loan amounts to monthly profit and verifying finances and collateral.
L006: This weaker application identifies red flags, such as Kofi has not started any of the proposed businesses, has no collateral, no proven income or profir, and a lot of other information. This system also requested business plans and some financial projections before proceeding.



2. We forbid this because it may lead to an incorrect decision being taken where we allow the model to make loan decisions whilst it does not have access to all information required for an efficient loan decision. Additionally, giving AI the sole power to approve or reject people's real life loan situations is quite unethical and requires some form of human interference.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** 357c722e07e36031e68e537231c24d0bc2c0dc2e

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [8]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).
fields = [ "applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs", "has_collateral_or_guarantor","repayment_months"]
compar = []
for field in fields:
    row = {"field": field}
    corr_count = 0
    for letter_id, gold_values in GOLD.items():
        extracted = results_frame[results_frame["letter_id"] == letter_id].iloc[0]
        predicted = extracted[field]
        actual = gold_values[field]
        if field == "applicant_name":
            correct = str(predicted).lower() == str(actual).lower()
        else:
            if pd.isna(actual) and pd.isna(predicted):
                correct = True
            else:
                correct = predicted == actual
        row[letter_id] = "Correct" if correct else "Wrong"
        if correct:
            corr_count += 1
    row["accuracy"] = f"{corr_count}/3 ({corr_count/3:.0%})"
    compar.append(row)

acc_df = pd.DataFrame(compar)

display(acc_df)
# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

,field,L001,L003,L006,accuracy
0,applicant_name,Correct,Correct,Correct,3/3 (100%)
1,amount_ghs,Correct,Correct,Correct,3/3 (100%)
2,purpose,Wrong,Wrong,Wrong,0/3 (0%)
3,monthly_profit_ghs,Correct,Correct,Correct,3/3 (100%)
4,has_collateral_or_guarantor,Correct,Correct,Correct,3/3 (100%)
5,repayment_months,Correct,Correct,Correct,3/3 (100%)


### Part 4.2 — Reliability: is the system consistent?

In [10]:
import json
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.
def extract_fields(letter_text, temperature=0):
    prompt = EXTRACT_PROMPT.format(letter_text=letter_text)
    result = ask_llm(prompt,system_prompt="You are a careful information extraction assistant.",temperature=temperature,max_tokens=500)
    try:
        cleaned = result.strip()
        if cleaned.startswith("```json"):
            cleaned = cleaned[7:]
        if cleaned.endswith("```"):
            cleaned = cleaned[:-3]
        return json.loads(cleaned.strip())
    except Exception as e:
        print("Warning: Could not parse JSON:", e)
        return None

for temperature in [0, 1.0]:
    results = []
    print(f"\n TEMPERATURE {temperature} ")
    for i in range(5):
        result = extract_fields(LETTERS["L004"], temperature=temperature)
        results.append(result)
        print(f"\nRun {i+1}:")
        print(result)

    v_results = [r for r in results if r is not None]
    u_results = set(
        json.dumps(r, sort_keys=True)
        for r in v_results
    )

    print("\nValid JSON:", len(v_results), "/ 5")
    print("Identical values across runs:",
          len(u_results) == 1)
    print("Number of unique outputs:", len(u_results))


 TEMPERATURE 0 
Token usage is  CompletionUsage(completion_tokens=70, prompt_tokens=505, total_tokens=575, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.135056118, prompt_time=0.052244463, completion_time=0.105816805, total_time=0.158061268)

Run 1:
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Token usage is  CompletionUsage(completion_tokens=70, prompt_tokens=505, total_tokens=575, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.007982094, prompt_time=0.025863631, completion_time=0.104936968, total_time=0.130800599)

Run 2:
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Token usage is  CompletionUsage(completion_tokens=70, prompt_tokens=505, total_tokens=575, co

### Part 4.3 — Hallucination probing

In [11]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?

adversarial_letter = LETTERS["L002"]

question = """What is the applicant's credit score?The answer must be based only on the information in the application.
If the information is not provided, explicitly say that it is not stated.
"""

output = ask_llm(f"{question}\n\n{adversarial_letter}",system_prompt=SUMMARY_SYSTEM_V2,temperature=0)

print("ADVERSARIAL TEST 1 ")
print(output)
weather_text = """Today's weather report indicates partly cloudy skies with temperatures
between 24 and 29 degrees Celsius. Rain is expected in some areas during
the afternoon.
"""
output = extract_fields(weather_text)
print("ADVERSARIAL TEST 2")
print(output)

Token usage is  CompletionUsage(completion_tokens=73, prompt_tokens=211, total_tokens=284, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.008166335, prompt_time=0.01088566, completion_time=0.188165256, total_time=0.199050916)
ADVERSARIAL TEST 1 
The applicant's credit score is not stated in the application. Kwame Boateng is applying for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. He mentions that business has been slow, but expects it to pick up after the festive season. The applicant does not provide any information about his credit history or score.
Token usage is  CompletionUsage(completion_tokens=55, prompt_tokens=429, total_tokens=484, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.008508289, prompt_time=0.059888366, completion_time=0.07167924, total_time=0.131567606)
ADVERSARIAL TEST 2
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guar

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.